# Loading Data with TimeToAlign!

This tutorial introduces the Loader pattern and EventStores - the foundation for bringing music data into TimeToAlign!

**Learning Objectives:**
- Use Loaders to ingest music data from various formats
- Navigate EventStores and access event data
- Understand the harmonized schema that unifies different data sources

**Prerequisites:**
- Basic Python and pandas knowledge
- TimeToAlign! installed (`pip install timetoalign`)

## Why Loaders Matter

Music data comes in many formats: MusicXML, MIDI, MEI, Humdrum, proprietary TSV exports, and more. Each format has its own structure, terminology, and quirks.

**The problem:** Without a unified approach, you'd need format-specific code for every data source, making cross-format analysis difficult and error-prone.

**The TimeToAlign! solution:** Loaders normalize heterogeneous formats into a consistent `EventStore`, enabling downstream processing without format-specific code.

```
MusicXML ─┐
MIDI ─────┼──> Loader ──> EventStore ──> DataFrame
TSV ──────┘
```

## Setup

In [1]:
# Standard imports
from pathlib import Path
import pandas as pd

# TimeToAlign! loaders
from timetoalign.loader.score.partitura import PartituraLoader
from timetoalign.loader.score.music21 import Music21Loader
from timetoalign.loader.score.tsv import TSVLoader

# Data directory - adjust if running from a different location
DATA_DIR = Path(".").resolve().parents[1] / "tests" / "data" / "midi" / "score"
assert DATA_DIR.is_dir(), f"Data directory not found: {DATA_DIR}"

# Our test piece: Chopin Etude Op.10 No.3
CHOPIN_XML = DATA_DIR / "chopin_op10_no3.musicxml"
CHOPIN_TSV = DATA_DIR / "ms3" / "chopin_op10_no3.notes.tsv"

print(f"MusicXML file: {CHOPIN_XML.name}")
print(f"TSV file: {CHOPIN_TSV.name}")

MusicXML file: chopin_op10_no3.musicxml
TSV file: chopin_op10_no3.notes.tsv


## The Loader Pattern

All TimeToAlign! loaders follow the same three-step pattern:

1. **Create** a loader instance
2. **Load** a file using `.load(path)`
3. **Access** the bundle containing EventStores

Let's see this in action with three different loaders, all loading the same Chopin piece:

In [2]:
# TSV Loader - for MS3 annotation exports
tsv_loader = TSVLoader()
tsv_loader.load(CHOPIN_TSV)
tsv_bundle = tsv_loader.bundle

print(f"TSV Loader bundle type: {type(tsv_bundle).__name__}")

TSV Loader bundle type: ScoreBundle


In [3]:
# Partitura Loader - for MusicXML, MIDI, and other formats
partitura_loader = PartituraLoader()
partitura_loader.load(CHOPIN_XML)
partitura_bundle = partitura_loader.bundle

print(f"Partitura Loader bundle type: {type(partitura_bundle).__name__}")

/home/laser/miniconda3/envs/timetoalign/lib/python3.11/site-packages/partitura/directions.py:514: UserWarning: error parsing "Lento, ma non troppo" (AttributeError)
  warnings.warn('error parsing "{}" ({})'.format(string, type(e).__name__))
/home/laser/miniconda3/envs/timetoalign/lib/python3.11/site-packages/partitura/directions.py:514: UserWarning: error parsing "cresc." (AttributeError)
  warnings.warn('error parsing "{}" ({})'.format(string, type(e).__name__))
/home/laser/miniconda3/envs/timetoalign/lib/python3.11/site-packages/partitura/directions.py:514: UserWarning: error parsing "ritenuto" (AttributeError)
  warnings.warn('error parsing "{}" ({})'.format(string, type(e).__name__))
/home/laser/miniconda3/envs/timetoalign/lib/python3.11/site-packages/partitura/directions.py:514: UserWarning: error parsing "a tempo" (AttributeError)
  warnings.warn('error parsing "{}" ({})'.format(string, type(e).__name__))


Partitura Loader bundle type: ScoreBundle


In [4]:
# Music21 Loader - alternative parser with different strengths
music21_loader = Music21Loader()
music21_loader.load(CHOPIN_XML)
music21_bundle = music21_loader.bundle

print(f"Music21 Loader bundle type: {type(music21_bundle).__name__}")

Music21 Loader bundle type: ScoreBundle


## Cross-Loader Validation

One of the key benefits of TimeToAlign! is that different loaders produce comparable output. Let's verify that all three loaders found the same number of notes:

In [5]:
# Convert to DataFrames and count notes
tsv_df = tsv_bundle.notes.to_dataframe()
partitura_df = partitura_bundle.notes.to_dataframe()
music21_df = music21_bundle.notes.to_dataframe()

# Count only Note events (not rests or other event types)
tsv_notes = len(tsv_df[tsv_df["event_type"] == "Note"])
partitura_notes = len(partitura_df[partitura_df["event_type"] == "Note"])
music21_notes = len(music21_df[music21_df["event_type"] == "Note"])

print(f"TSV Loader:       {tsv_notes} notes")
print(f"Partitura Loader: {partitura_notes} notes")
print(f"Music21 Loader:   {music21_notes} notes")

# Validation - all should match the gold standard count
assert tsv_notes == 498, f"Expected 498 notes from TSV, got {tsv_notes}"
assert partitura_notes == 498, f"Expected 498 notes from Partitura, got {partitura_notes}"
assert music21_notes == 498, f"Expected 498 notes from Music21, got {music21_notes}"

print("\nAll loaders agree: 498 notes in Chopin Op.10 No.3")

TSV Loader:       498 notes
Partitura Loader: 498 notes
Music21 Loader:   498 notes

All loaders agree: 498 notes in Chopin Op.10 No.3


## The EventStore

Each bundle contains one or more **EventStores** - efficient, PyArrow-backed tables that hold musical events.

Key characteristics:
- **High Performance**: Built on Apache Arrow for fast columnar operations
- **Type Safety**: Schema metadata preserves units and types
- **Pandas Interop**: Easy conversion with `.to_dataframe()`

In [6]:
# Access the notes EventStore
notes_store = tsv_bundle.notes

print(f"EventStore type: {type(notes_store).__name__}")
print(f"Number of events: {len(notes_store)}")
print(f"\nUnderlying storage: {type(notes_store.table).__name__}")

EventStore type: NoteEventStore
Number of events: 498

Underlying storage: Table


In [7]:
# Examine the schema
schema = notes_store.table.schema

print("EventStore Schema:")
print("="*50)
for field in schema:
    metadata = field.metadata if field.metadata else {}
    meta_str = ", ".join(f"{k.decode()}={v.decode()}" for k, v in metadata.items()) if metadata else "(no metadata)"
    print(f"  {field.name:20} {str(field.type):25} {meta_str}")

EventStore Schema:
  id                   string                    (no metadata)
  name                 string                    (no metadata)
  temporal_type        string                    (no metadata)
  event_type           string                    (no metadata)
  start                struct<value: double not null, numerator: int64, denominator: int64> unit=quarters
  end                  struct<value: double not null, numerator: int64, denominator: int64> unit=quarters
  duration             struct<value: double not null, numerator: int64, denominator: int64> unit=quarters
  duration_float       double                    (no metadata)
  mc                   int64                     number_type=int64
  mn                   string                    (no metadata)
  mc_onset             struct<num: int64 not null, den: int64 not null> number_type=fraction
  mn_onset             struct<num: int64 not null, den: int64 not null> number_type=fraction
  midi_pitch           struct<ep

## The Harmonized Schema

TimeToAlign! uses a harmonized schema to represent events consistently across formats. The key columns are:

| Column | Description |
|--------|-------------|
| `id` | Unique identifier for the event |
| `temporal_type` | "instant" or "interval" |
| `event_type` | Type of event (Note, Rest, etc.) |
| `start`, `end`, `duration` | Temporal coordinates (as structs) |
| `duration_float` | Duration as a float for quick queries |
| `mc`, `mn` | Measure count and measure number |
| `midi_pitch` | MIDI pitch number (0-127) |
| `spelled_pitch` | Pitch spelling information |

Let's examine a few events:

In [8]:
# Show selected columns for the first few notes
display_cols = ["id", "name", "temporal_type", "event_type", "duration_float", "mc", "mn", "midi_pitch", "octave"]
tsv_df[display_cols].head(10)

,id,name,temporal_type,event_type,duration_float,mc,mn,midi_pitch,octave
0,note_0.0_0,B3,interval,Note,0.50,1,1,"{'ep': 59, 'epc': 11}",3
1,note_0.5_1,E2,interval,Note,0.25,2,2,"{'ep': 40, 'epc': 4}",2
2,note_0.5_2,E2,interval,Note,1.00,2,2,"{'ep': 40, 'epc': 4}",2
3,note_0.5_3,G#3,interval,Note,0.25,2,2,"{'ep': 56, 'epc': 8}",3
4,note_0.5_4,E4,interval,Note,0.50,2,2,"{'ep': 64, 'epc': 4}",4
5,note_0.75_5,B2,interval,Note,0.50,2,2,"{'ep': 47, 'epc': 11}",2
6,note_0.75_6,B3,interval,Note,0.25,2,2,"{'ep': 59, 'epc': 11}",3
7,note_1.0_7,G#3,interval,Note,0.25,2,2,"{'ep': 56, 'epc': 8}",3
8,note_1.0_8,D#4,interval,Note,0.25,2,2,"{'ep': 63, 'epc': 3}",4
9,note_1.25_9,B2,interval,Note,0.25,2,2,"{'ep': 47, 'epc': 11}",2


## Pitch Information

The `spelled_pitch` column contains rich pitch information as a struct. This preserves the enharmonic spelling (e.g., G# vs Ab) which is lost when using only MIDI pitch numbers.

In [9]:
# Extract spelled pitch information
first_note = tsv_df.iloc[0]

print(f"First note: {first_note['name']}")
print(f"MIDI pitch: {first_note['midi_pitch']}")
print(f"Octave: {first_note['octave']}")
print(f"\nSpelled pitch struct:")

spelled = first_note['spelled_pitch']
if isinstance(spelled, dict):
    for key, value in spelled.items():
        print(f"  {key}: {value}")

First note: B3
MIDI pitch: {'ep': 59, 'epc': 11}
Octave: 3

Spelled pitch struct:
  gpc_int: 6
  gpc_str: B
  acc: 0
  spc_int: 5
  spc_str: B
  sp: B3
  cents: 0.0


## Duration Analysis

TimeToAlign! stores durations in quarter notes. Let's analyze the rhythmic content of our piece:

In [10]:
# Analyze duration distribution
duration_counts = tsv_df['duration_float'].value_counts().sort_index()

print("Duration Distribution (in quarter notes):")
print("="*40)
for duration, count in duration_counts.items():
    bar = "*" * min(int(count/5), 50)
    print(f"  {duration:6.3f}: {count:4d} {bar}")

Duration Distribution (in quarter notes):
   0.000:    4 
   0.250:  391 **************************************************
   0.500:   42 ********
   0.750:    2 
   1.000:   54 **********
   2.000:    5 *


In [11]:
# Summary statistics
print("Duration Statistics:")
print(f"  Shortest note: {tsv_df['duration_float'].min():.3f} quarters")
print(f"  Longest note:  {tsv_df['duration_float'].max():.3f} quarters")
print(f"  Mean duration: {tsv_df['duration_float'].mean():.3f} quarters")

Duration Statistics:
  Shortest note: 0.000 quarters
  Longest note:  2.000 quarters
  Mean duration: 0.370 quarters


## Navigating by Measure

The `mc` (measure count) and `mn` (measure number) columns allow easy navigation through the score:

In [14]:
# Notes per measure
notes_per_measure = tsv_df.groupby('mn').size()

print("Notes per Measure:")
print("="*40)
for measure, count in notes_per_measure.items():
    bar = "*" * min(count, 50)
    print(f"  m.{measure:<2}: {count:3d} {bar}")

Notes per Measure:
  m.1 :   1 *
  m.10:  22 **********************
  m.11:  24 ************************
  m.12:  22 **********************
  m.13:  25 *************************
  m.14:  25 *************************
  m.15:  28 ****************************
  m.16:  27 ***************************
  m.17:  56 **************************************************
  m.18:  18 ******************
  m.19:  21 *********************
  m.2 :  21 *********************
  m.20:  21 *********************
  m.21:  17 *****************
  m.22:   7 *******
  m.3 :  24 ************************
  m.4 :  22 **********************
  m.5 :  25 *************************
  m.6 :  25 *************************
  m.7 :  24 ************************
  m.8 :  21 *********************
  m.9 :  22 **********************


In [15]:
# Get all notes in a specific measure
measure_5 = tsv_df[tsv_df['mn'] == 5]

print(f"Notes in Measure 5: {len(measure_5)}")
print()
measure_5[['name', 'duration_float', 'voice', 'staff']].head(10)

Notes in Measure 5: 0



,name,duration_float,voice,staff


## Comparing Loader Outputs

While all loaders produce the same number of notes, there can be subtle differences in how they interpret the score. Let's compare the first few notes:

In [16]:
# Compare ID schemes and names
comparison = pd.DataFrame({
    'TSV_id': tsv_df['id'].head(5).values,
    'TSV_name': tsv_df['name'].head(5).values,
    'Partitura_id': partitura_df['id'].head(5).values,
    'Music21_id': music21_df['id'].head(5).values[:5],
})

print("Comparing first 5 notes across loaders:")
comparison

Comparing first 5 notes across loaders:


,TSV_id,TSV_name,Partitura_id,Music21_id
0,note_0.0_0,B3,n0,140255202866064
1,note_0.5_1,E2,n1,140255202479312
2,note_0.5_2,E2,n2,140255201767632
3,note_0.5_3,G#3,n3,140255201805648
4,note_0.5_4,E4,n4,140255201810256


## Unit Metadata

TimeToAlign! stores unit information in the PyArrow schema metadata. This ensures coordinates are always interpreted correctly:

In [17]:
# Extract unit metadata for temporal columns
temporal_columns = ['start', 'end', 'duration']

print("Unit Metadata for Temporal Columns:")
print("="*40)
for field in notes_store.table.schema:
    if field.name in temporal_columns and field.metadata:
        unit = field.metadata.get(b'unit', b'(unknown)').decode()
        print(f"  {field.name}: {unit}")

Unit Metadata for Temporal Columns:
  start: quarters
  end: quarters
  duration: quarters


## Voice and Staff Information

For piano music, notes are distributed across staves and voices:

In [18]:
# Analyze voice and staff distribution
staff_voice = tsv_df.groupby(['staff', 'voice']).size().unstack(fill_value=0)

print("Notes by Staff and Voice:")
print()
staff_voice

Notes by Staff and Voice:



voice,1,2,3
staff,,,
1,112,50,153
2,142,2,39


## Summary

In this tutorial, we learned:

1. **The Loader Pattern**: Create -> Load -> Access Bundle
2. **Three Score Loaders**: TSVLoader, PartituraLoader, Music21Loader
3. **EventStore**: PyArrow-backed, high-performance event storage
4. **Harmonized Schema**: Consistent columns across all loaders
5. **Cross-Validation**: Same piece from different sources yields same note count

**Key Takeaway:**
> Loaders normalize heterogeneous formats into a consistent EventStore, enabling downstream processing without format-specific code.

## Next Steps

- **03_conversion_maps.ipynb**: Learn how to convert between coordinate systems
- **04_building_timelines.ipynb**: Create Timeline objects from EventStores

---

## Exercise: Load Another Score

**Task:** Load the Beethoven String Quartet from `beethoven_op18.mid` and analyze its structure.

**Hints:**
1. Use `PartituraLoader` for MIDI files
2. Check how many parts are in the score
3. Count notes per part

<details>
<summary>Solution</summary>

```python
# Load the Beethoven quartet
beethoven_path = DATA_DIR / "beethoven_op18.mid"
loader = PartituraLoader()
loader.load(beethoven_path)
beethoven_bundle = loader.bundle

# Analyze
df = beethoven_bundle.notes.to_dataframe()
print(f"Total notes: {len(df)}")
print(f"\nNotes per part:")
print(df.groupby('part_id').size())
```

</details>